# KURE-v1 임베딩 (코랩 GPU)

신한은행 상품 RAG의 청크·공통 조항·용어집과 검색 평가 질문을 **nlpai-lab/KURE-v1**(bge-m3 한국어 미세조정, 1024차원, MIT)로 임베딩합니다.

**준비**
1. 런타임 → 런타임 유형 변경 → **GPU (T4 이상)**
2. 로컬에서 만든 입력 파일 `data/colab/embed_input.jsonl` (로컬: `python analysis/s10_chunks.py export-colab`)

**결과**: `embed_output.jsonl` (한 줄 = `{hash, model, vec}`) → 다운로드해서 로컬 `data/colab/`에 두기

> 입력 파일에는 상품 문서 본문이 들어 있으니, 다 쓰면 코랩 런타임을 삭제(런타임 → 런타임 연결 해제 및 삭제)하세요.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -U sentence-transformers

## 1. 입력 파일 올리기
아래 셀을 실행하고 `embed_input.jsonl`을 선택합니다. (구글 드라이브에 둔 경우 두 번째 방법)

In [ ]:
import json, os
from google.colab import files

INPUT = "embed_input.jsonl"
if not os.path.exists(INPUT):
    up = files.upload()            # embed_input.jsonl 선택
    INPUT = next(iter(up))

# (드라이브를 쓰려면) from google.colab import drive; drive.mount('/content/drive'); INPUT = '/content/drive/MyDrive/embed_input.jsonl'

rows = [json.loads(l) for l in open(INPUT, encoding="utf-8")]
print(f"전체 {len(rows):,}개 — 글 {sum(r['kind']=='passage' for r in rows):,}개 / 질문 {sum(r['kind']=='query' for r in rows):,}개")
print(rows[0]["text"][:200])

## 2. 모델 불러오기
KURE-v1은 모델 카드상 질문·문서 앞에 붙이는 지시문(prefix)이 없습니다. 같은 방식으로 encode 합니다.

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

MODEL = "nlpai-lab/KURE-v1"
model = SentenceTransformer(MODEL, device="cuda")
model.half()                       # T4에서 빠르게 (fp16)
model.max_seq_length = 2048        # 우리 청크는 최대 약 1,500자 + 머리말 → 2048 토큰이면 충분
print(model.get_sentence_embedding_dimension(), model.max_seq_length)

## 3. 임베딩
길이순으로 정렬해 배치를 만들면 빠릅니다. 코사인 유사도를 쓰므로 정규화해서 저장합니다.

In [ ]:
import numpy as np, time

order = sorted(range(len(rows)), key=lambda i: len(rows[i]["text"]))
texts = [rows[i]["text"] for i in order]
t0 = time.time()
vecs = model.encode(texts, batch_size=32, normalize_embeddings=True, show_progress_bar=True, convert_to_numpy=True)
print(f"{len(texts):,}개, {time.time()-t0:.0f}초, 모양 {vecs.shape}")
emb = np.zeros_like(vecs)
emb[order] = vecs
assert emb.shape[1] == 1024 and np.isfinite(emb).all()

## 4. 확인: 질문 몇 개로 가장 가까운 글 보기
(로컬의 정식 평가는 상품명 좁히기·키워드·재순위까지 포함한 `s12_retrieval.py`로 합니다. 여기서는 벡터가 정상인지만 봅니다)

In [ ]:
P = [i for i, r in enumerate(rows) if r["kind"] == "passage"]
Q = [i for i, r in enumerate(rows) if r["kind"] == "query"]
for qi in Q[:3]:
    sims = emb[P] @ emb[qi]
    top = np.argsort(-sims)[:3]
    print("Q:", rows[qi]["text"])
    for t in top:
        print(f"   {sims[t]:.3f}", rows[P[t]]["text"][:80].replace("\n", " / "))

## 5. 저장하고 내려받기

In [ ]:
OUT = "embed_output.jsonl"
with open(OUT, "w", encoding="utf-8") as f:
    for r, v in zip(rows, emb):
        f.write(json.dumps({"hash": r["hash"], "model": MODEL, "vec": [round(float(x), 6) for x in v]}) + "\n")
print(f"{OUT}: {os.path.getsize(OUT)/1e6:.1f}MB")
files.download(OUT)

## 6. 로컬에서 할 일
`embed_output.jsonl`을 `최종프로젝트/data/colab/`에 두고:

1. `.env` 바꾸기: `EMBED_MODEL=nlpai-lab/KURE-v1`, `EMBED_DIM=1024`
2. `python analysis/s10_chunks.py import-colab data/colab/embed_output.jsonl` — 해시·개수·차원 확인 후 캐시에 합침
3. `python analysis/s9_load_db.py` — 스키마를 1024차원으로 다시 만듦
4. `python analysis/s10_chunks.py load` — 청크·공통 조항·용어 벡터 적재
5. `python analysis/s12_retrieval.py dev routed rerank` — OpenAI 임베딩과 같은 평가셋으로 비교

OpenAI 결과도 캐시에 남아 있어서, `.env`만 되돌리면 언제든 다시 비교할 수 있습니다.